In [21]:
import pandas as pd
import numpy as np
np.random.seed(42)
import random
random.seed(42)

from pdb import set_trace

from statistics import median, mean

from sklearn.cluster import DBSCAN
from sklearn.feature_extraction.text import CountVectorizer

from gensim.parsing.preprocessing import lower_to_unicode, preprocess_string, strip_tags, strip_punctuation, strip_multiple_whitespaces, strip_numeric

from tqdm.auto import tqdm
tqdm.pandas()

In [22]:
# --- Replacement for Cell 2 ---

from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR  = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from PyDI.io import load_json

# Load your 4 datasets
products_1_cleaned = load_json(OUTPUT_DIR / "normalized_products_after_drp_cols_and_extractions" / "dataset_1_normalized.json")
products_2_cleaned = load_json(OUTPUT_DIR / "normalized_products_after_drp_cols_and_extractions" / "dataset_2_normalized.json")
products_3_cleaned = load_json(OUTPUT_DIR / "normalized_products_after_drp_cols_and_extractions" / "dataset_3_normalized.json")
products_4_cleaned = load_json(OUTPUT_DIR / "normalized_products_after_drp_cols_and_extractions" / "dataset_4_normalized.json")

local_norm_datasets = [products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned]

# Cast IDs to int64 for consistency
for df in local_norm_datasets:
    df['id'] = df['id'].astype('int64')

# Combine into one universe and drop duplicate record IDs across files
corpus = pd.concat(local_norm_datasets, ignore_index=True).drop_duplicates(subset=['id'])
print(f"Total unique IDs in universe: {len(corpus)}")

Total unique IDs in universe: 3012


In [23]:
# --- Replacement for Cell 3 ---
# Filter clusters based on size (e.g., only entities with > 3 records) #orignal 3 now trying 2
counts = corpus['cluster_id'].value_counts()
relevant_cluster_ids = counts[counts > 2].index

# Select one representative per relevant cluster
corpus_selection = corpus[corpus['cluster_id'].isin(relevant_cluster_ids)].copy()
corpus_selection = corpus_selection.drop_duplicates('cluster_id')

# Preprocessing for Vectorization
CUSTOM_FILTERS = [lambda x: x.lower(), strip_tags, strip_punctuation, strip_multiple_whitespaces]
corpus_selection['title_processed'] = corpus_selection['title'].apply(lower_to_unicode)
corpus_selection['title_processed'] = corpus_selection['title_processed'].apply(preprocess_string, args=(CUSTOM_FILTERS,))
corpus_selection['title_processed'] = corpus_selection['title_processed'].apply(lambda x: ' '.join(x))

# Load cleansed PDC2020 corpus

In [24]:
# corpus = pd.read_pickle('../../../data/interim/wdc-lspc/corpus/dedup_preprocessed_lspcV2020_only_en_strict_only_long_title_only_mainentity.pkl.gz')
# corpus.head()

In [25]:
# counts = corpus['cluster_id'].value_counts()
# counts = counts[counts > 3]

# Apply DBSCAN clustering and save it for manual labeling

1-DBSCAN to cluster unique clusters using cluser_ids based on their title (unique entities which are 812 for thsi dataset) we 

2-Seen entities (entites with 4-20 records/common) used for training

3-Unseen entities (entities that only appear in 2-3 records) used for testing. change to 1 to 3

In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer

eps_list = [0.3] #lowered epsilon from 0.35 so that it is harder but now we will have less data
min_samples_list = [1] #same as repo

#create dbscan folder to have seen and unseen subfolders
dbscan_folder = OUTPUT_DIR / "dbscan_results"
dbscan_folder.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = dbscan_folder

# This list now contains EVERY column from your provided index
# Organized for readability: Identifiers -> Technical Specs -> Metadata
full_column_list = [
    'dbscan_cluster', 'id', 'cluster_id', 'product_type','brand', 'title', 'model', 'model_number',
    'description','chipset_name', 'vram_gb', 'storage_gb', 'read_speed_mb_s', 
    'write_speed_mb_s', 'bus_type', 'interface_type', 'storage_connection_type', 
    'memory_type', 'color', 'form_factor', 'width_mm', 'length_mm', 'height_mm', 
    'weight_g', 'price', 'priceCurrency' , 'url', 'title_description'
]

for eps in eps_list:
    for min_sample in min_samples_list:
        print(f'eps: {eps}, min_samples: {min_sample}')
        
        corpus_selection = corpus[corpus['cluster_id'].isin(counts.index)].copy()
        corpus_selection = corpus_selection.drop_duplicates('cluster_id')
        CUSTOM_FILTERS = [lambda x: x.lower(), strip_tags, strip_punctuation, strip_multiple_whitespaces]

        corpus_selection['title_processed'] = corpus_selection['title'].apply(lower_to_unicode)
        corpus_selection['title_processed'] = corpus_selection['title_processed'].apply(preprocess_string, args=(CUSTOM_FILTERS,))
        corpus_selection['title_processed'] = corpus_selection['title_processed'].apply(lambda x: ' '.join(x))
        

        # vectorizer = TfidfVectorizer(ngram_range=(1, 3), analyzer='char_wb', min_df=1) #will test tf-idf 

        vectorizer = CountVectorizer(strip_accents='unicode', binary=True, min_df=2) # Changed to 2. for the big WDC dataset was 4
        matrix = vectorizer.fit_transform(corpus_selection['title_processed'])

        dbscan = DBSCAN(metric='cosine', eps=eps, min_samples=min_sample)
        clustering = dbscan.fit(matrix)
        corpus_selection['dbscan_cluster'] = clustering.labels_
        
        counts_relevant = corpus['cluster_id'].value_counts()

        # Seen/Unseen logic based on cluster frequency (how the wdc repo did it, but doesnt work for my smaller subset of course)
        # counts_relevant_unseen = counts_relevant[(counts_relevant > 3) & (counts_relevant < 7)]
        # counts_relevant_seen = counts_relevant[(counts_relevant > 6) & (counts_relevant < 81)]
        
        
        # Adjusted logic for a dataset of ~3,000 IDs
        # Unseen: Entities with exactly 2 records (The "Tail" of your data)
        counts_relevant_unseen = counts_relevant[(counts_relevant >= 1) & (counts_relevant <= 3)] #from 1 to 3 instead of 2 to 3 to get more unseen data

        # Seen: Entities with 4to 21 records (The "Head" of your data)
        counts_relevant_seen = counts_relevant[(counts_relevant >= 4) & (counts_relevant < 21)]


        # --- SEEN DATA PROCESSING ---
        print(f'Seen data:')
        corpus_selection_seen = corpus_selection[corpus_selection['cluster_id'].isin(counts_relevant_seen.index)].copy()
        corpus_selection_seen = corpus_selection_seen[corpus_selection_seen['dbscan_cluster'] != -1]
        
        counts_clustering_seen = corpus_selection_seen['dbscan_cluster'].value_counts()
        counts_clustering_seen = counts_clustering_seen[counts_clustering_seen > 1] # might Change neighborhood size from > 2 to > 1 to allow for prod_1_to_prodx to match
        corpus_selection_seen = corpus_selection_seen[corpus_selection_seen['dbscan_cluster'].isin(counts_clustering_seen.index)]
        corpus_selection_seen = corpus_selection_seen.sort_values('dbscan_cluster')
        
        # Using all your columns here
        corpus_selection_seen = corpus_selection_seen[full_column_list]
        corpus_selection_seen.to_excel(OUTPUT_DIR / f'seen_dbscan_manual_labeling.xlsx', index=False)
        
        db_clu_seen = corpus_selection_seen[['cluster_id', 'dbscan_cluster']].drop_duplicates('cluster_id')
        db_clu_seen.to_csv(OUTPUT_DIR / 'seen_dbscan_mapping.csv', index=False)
        
        # --- UNSEEN DATA PROCESSING ---
        print(f'Unseen data:')
        corpus_selection_unseen = corpus_selection[corpus_selection['cluster_id'].isin(counts_relevant_unseen.index)].copy()
        corpus_selection_unseen = corpus_selection_unseen[corpus_selection_unseen['dbscan_cluster'] != -1]
        
        counts_clustering_unseen = corpus_selection_unseen['dbscan_cluster'].value_counts()
        counts_clustering_unseen = counts_clustering_unseen[counts_clustering_unseen > 1] # might Change neighborhood size from > 2 to > 1 to allow for prod_1_to_prodx to match
        corpus_selection_unseen = corpus_selection_unseen[corpus_selection_unseen['dbscan_cluster'].isin(counts_clustering_unseen.index)]
        corpus_selection_unseen = corpus_selection_unseen.sort_values('dbscan_cluster')
        
        corpus_selection_unseen = corpus_selection_unseen[full_column_list]
        corpus_selection_unseen.to_excel(OUTPUT_DIR / f'unseen_dbscan_manual_labeling.xlsx', index=False)
        
        db_clu_unseen = corpus_selection_unseen[['cluster_id', 'dbscan_cluster']].drop_duplicates('cluster_id')
        db_clu_unseen.to_csv(OUTPUT_DIR / 'unseen_dbscan_mapping.csv', index=False)

        print(f'-------------------------------------------------------------------------')

eps: 0.3, min_samples: 1
Seen data:
Unseen data:
-------------------------------------------------------------------------


## old setup from original WDC REPO

In [ ]:
# eps_list = [0.35]
# min_samples_list = [1]

# for eps in eps_list:
#     for min_sample in min_samples_list:
#         print(f'eps: {eps}, min_samples: {min_sample}')
        
#         corpus_selection = corpus[corpus['cluster_id'].isin(counts.index)].copy()
#         corpus_selection = corpus_selection.drop_duplicates('cluster_id')
#         CUSTOM_FILTERS = [lambda x: x.lower(), strip_tags, strip_punctuation, strip_multiple_whitespaces]

#         corpus_selection['title_processed'] = corpus_selection['title'].apply(lower_to_unicode)
#         corpus_selection['title_processed'] = corpus_selection['title_processed'].apply(preprocess_string, args=(CUSTOM_FILTERS,))
#         corpus_selection['title_processed'] = corpus_selection['title_processed'].apply(lambda x: ' '.join(x))
        
#         vectorizer = CountVectorizer(strip_accents='unicode', binary=True, min_df=4)
#         #vectorizer = TfidfVectorizer(strip_accents='unicode', use_idf=False)
#         matrix = vectorizer.fit_transform(corpus_selection['title_processed'])

#         dbscan = DBSCAN(metric='cosine', eps=eps, min_samples=min_sample)
#         #dbscan = OPTICS(metric='cosine', max_eps=eps, eps=eps, min_samples=min_sample, cluster_method='dbscan')
#         clustering = dbscan.fit(matrix)
#         corpus_selection['dbscan_cluster'] = clustering.labels_
        
#         counts_relevant = corpus['cluster_id'].value_counts()

#         counts_relevant_unseen = counts_relevant[counts_relevant > 3]
#         counts_relevant_unseen = counts_relevant_unseen[counts_relevant_unseen < 7]
        
#         counts_relevant_seen = counts_relevant[counts_relevant > 6]
#         counts_relevant_seen = counts_relevant_seen[counts_relevant_seen < 81]
        
#         print(f'Seen data:')
#         corpus_selection_seen = corpus_selection[corpus_selection['cluster_id'].isin(counts_relevant_seen.index)].copy()
#         corpus_selection_seen = corpus_selection_seen[corpus_selection_seen['dbscan_cluster'] != -1]
        
#         print(f'Clusters found: {len(corpus_selection_seen["dbscan_cluster"].unique())}')
#         print(f'Mean cluster size: {mean(corpus_selection_seen["dbscan_cluster"].value_counts())}, Median cluster_size: {median(corpus_selection_seen["dbscan_cluster"].value_counts())}')
        
#         counts_clustering = corpus_selection_seen['dbscan_cluster'].value_counts()
#         counts_clustering = counts_clustering[counts_clustering > 2]
#         corpus_selection_seen = corpus_selection_seen[corpus_selection_seen['dbscan_cluster'].isin(counts_clustering.index)]
#         corpus_selection_seen = corpus_selection_seen.sort_values('dbscan_cluster')
        
#         print(f'Clusters >2 found: {len(corpus_selection_seen["dbscan_cluster"].unique())}')
#         print(f'Mean cluster size: {mean(corpus_selection_seen["dbscan_cluster"].value_counts())}, Median cluster_size: {median(corpus_selection_seen["dbscan_cluster"].value_counts())}\n')
#         corpus_selection_seen = corpus_selection_seen[['dbscan_cluster', 'brand', 'title', 'description', 'price', 'priceCurrency',
#        'specTableContent', 'id', 'cluster_id', 'sku', 'mpn', 'gtin', 'gtin8',
#        'gtin12', 'gtin13', 'gtin14', 'productID', 'identifier']]
        
#         corpus_selection_seen.to_excel(f'../../../data/interim/wdc-lspc/corpus/seen_dbscan_eps{eps}_minsamples{min_sample}_dedup_preprocessed_lspcV2020_only_en_strict_only_long_title_only_mainentity.xlsx', header=True, index=False)
        
#         db_clu = corpus_selection_seen[['cluster_id', 'dbscan_cluster']].copy()
#         db_clu = db_clu.drop_duplicates('cluster_id')
#         db_clu.to_csv(f'../../../data/interim/wdc-lspc/corpus/seen_dbscan_mapping.csv', header=True, index=False)
#         db_clu = corpus_selection_seen['dbscan_cluster'].copy()
#         db_clu = db_clu.drop_duplicates()
#         db_clu = db_clu.sort_values()
#         db_clu.to_csv(f'../../../data/interim/wdc-lspc/corpus/seen_dbscan_clusters.csv', header=True, index=False)
        
#         print(f'Unseen data:')
#         corpus_selection_unseen = corpus_selection[corpus_selection['cluster_id'].isin(counts_relevant_unseen.index)].copy()
#         corpus_selection_unseen = corpus_selection_unseen[corpus_selection_unseen['dbscan_cluster'] != -1]
        
#         print(f'Clusters found: {len(corpus_selection_unseen["dbscan_cluster"].unique())}')
#         print(f'Mean cluster size: {mean(corpus_selection_unseen["dbscan_cluster"].value_counts())}, Median cluster_size: {median(corpus_selection_unseen["dbscan_cluster"].value_counts())}')
        
#         counts_clustering = corpus_selection_unseen['dbscan_cluster'].value_counts()
#         counts_clustering = counts_clustering[counts_clustering > 2]
#         corpus_selection_unseen = corpus_selection_unseen[corpus_selection_unseen['dbscan_cluster'].isin(counts_clustering.index)]
#         corpus_selection_unseen = corpus_selection_unseen.sort_values('dbscan_cluster')
        
#         print(f'Clusters >2 found: {len(corpus_selection_unseen["dbscan_cluster"].unique())}')
#         print(f'Mean cluster size: {mean(corpus_selection_unseen["dbscan_cluster"].value_counts())}, Median cluster_size: {median(corpus_selection_unseen["dbscan_cluster"].value_counts())}\n')
#         corpus_selection_unseen = corpus_selection_unseen[['dbscan_cluster', 'brand', 'title', 'description', 'price', 'priceCurrency',
#        'specTableContent', 'id', 'cluster_id', 'sku', 'mpn', 'gtin', 'gtin8',
#        'gtin12', 'gtin13', 'gtin14', 'productID', 'identifier']]
        
#         corpus_selection_unseen.to_excel(f'../../../data/interim/wdc-lspc/corpus/unseen_dbscan_eps{eps}_minsamples{min_sample}_dedup_preprocessed_lspcV2020_only_en_strict_only_long_title_only_mainentity.xlsx', header=True, index=False)
        
#         db_clu = corpus_selection_unseen[['cluster_id', 'dbscan_cluster']].copy()
#         db_clu = db_clu.drop_duplicates('cluster_id')
#         db_clu.to_csv(f'../../../data/interim/wdc-lspc/corpus/unseen_dbscan_mapping.csv', header=True, index=False)
#         db_clu = corpus_selection_unseen['dbscan_cluster'].copy()
#         db_clu = db_clu.drop_duplicates()
#         db_clu = db_clu.sort_values()
#         db_clu.to_csv(f'../../../data/interim/wdc-lspc/corpus/unseen_dbscan_clusters.csv', header=True, index=False)

#         print(f'-------------------------------------------------------------------------')